In [37]:
from dotenv import load_dotenv
import os
load_dotenv()

GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')
connection = os.getenv("PG_CONNECTION_STRING")


In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_postgres.vectorstores import PGVector

raw_documents = TextLoader("data/how_to_read_pnl.txt").load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GoogleGeminiKey)
db = PGVector.from_documents(documents, embeddings, connection=connection)

In [12]:
# create retriever
retriever = db.as_retriever(search_kwargs={"k":10})

# fetch relevant documents.
docs = retriever.invoke("how to read pnl?")
docs

[Document(id='4c7a2f4d-0912-46ae-816b-744c83117aac', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='Tata McGraw Hill Publishing Company Limited\nNEW DELHI\nN. Ramachandran\nPrincipal Consultant\nManagement Advisory Services\nKochi\nRam Kumar Kakani\nAssociate Professor, XLRI\nJamshedpur\nHow to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw Hill Professional: Finance Made Easy Series\nPublished by Tata McGraw Hill Education Private Limited,\n7 West Patel Nagar, New Delhi 110 008.\nCopyright © 2010, by Tata McGraw Hill Education Private Limited\nNo part of this publication may be reproduced or distributed in any form or by any\nmeans, electronic, mechanical, photocopying, recording, or otherwise or stored in a\ndatabase or retrieval system without the prior written permission of the publishers. The\nprogram listings (if any) may be entered, stored and executed in a computer system,\nbut they may not be reproduced for publication.\nThis edition can be exported from Indi

In [13]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Markdown, display

llm = GoogleGenerativeAI(model="gemini-2.0-flash-001",api_key=GoogleGeminiKey,temperature=0)

prompt_template = ChatPromptTemplate.from_messages(
[("system", """
            Answer the question on the based on the given context below. If the question cannot be answered using the information
provided in the context, respond with "I don't know".
    """),
    ("system","Answer always in markdown format and List the items."),
    ("human", "Context: {context}"),
    ("human", "Question: {question}")])

chain = prompt_template | llm

query = "how to read pnl?"

docs = retriever.get_relevant_documents(query)

result = chain.invoke({"context": docs, "question": query})

display(Markdown(result))


C:\Users\jeetc\AppData\Local\Temp\ipykernel_9756\2773577904.py:20: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


Here are key points on how to read a Profit and Loss (P&L) statement, based on the provided context:

*   **Non-Operating Income:** This comes from activities not part of the core business.

*   **Profit After Tax (PAT) or Net Profit:** This is the net income after deducting tax from Profit Before Tax (PBT). It measures net earnings available to the company's shareholders.

*   **Profit Available for Distribution:** This is calculated by summing PAT and retained earnings from previous years.

*   **Contents of Profit and Loss Account (Flow Chart):**
    1.  Sales (or revenues)
    2.  Less: Cost of goods sold
    3.  Gross profit
    4.  Less: Operating expenses
    5.  Operating profit
    6.  Less: Non-operating expenses
    7.  Profit before interest and tax (PBIT)
    8.  Less: Interest
    9.  Profit before tax (PBT)
    10. Less: Tax
    11. Profit after tax (PAT)
    12. Add: Previous year’s balance of P&L A/C
    13. Profit available for distribution
    14. Less: Appropriations
    15. Retained earnings

In [20]:
from langchain_core.runnables import chain

@chain
def chat(input):
    # fetch relevant documents
    docs = retriever.get_relevant_documents(input)
    
    # formatted prompt 
    formatted = prompt_template.invoke({"context": docs, "question": input})

    # generate response
    answer = llm.invoke(formatted)

    return {"answer": answer, "docs": docs}


chat.invoke(query)

{'answer': "Here are key points on how to read a Profit and Loss (P&L) statement, based on the provided context:\n\n*   **Non-Operating Income:** This comes from activities not part of the core business.\n\n*   **Profit After Tax (PAT) or Net Profit:** This is the net income after deducting tax from Profit Before Tax (PBT). It measures net earnings available to the company's shareholders.\n\n*   **Profit Available for Distribution:** This is calculated by summing PAT and retained earnings from previous years.\n\n*   **Contents of Profit and Loss Account (Flow Chart):**\n    1.  Sales (or revenues)\n    2.  Less: Cost of goods sold\n    3.  Gross profit\n    4.  Less: Operating expenses\n    5.  Operating profit\n    6.  Less: Non-operating expenses\n    7.  Profit before interest and tax (PBIT)\n    8.  Less: Interest\n    9.  Profit before tax (PBT)\n    10. Less: Tax\n    11. Profit after tax (PAT)\n    12. Add: Previous year’s balance of P&L A/C\n    13. Profit available for distrib

#### $ Problems of to make production ready robust RAG chat application.
1. How do we handle the variability in the quality of a user input.
2. How do we route queries to retrieve relevant data from a variety of data sources.
3. How do we transform natural language to the query language of the target data source?
4. How do we optimize out indexing process ie embedding text splitting?
 

#### Query Transformation can solve the first problem.
### Rewrite-Retrieve-Read

In [33]:
query = """
Today I woke up and realized that my laptop battery was low.then I found a charger in the kitchen but it wasn't working properly.
So I decided to ask for help from my friend who is an engineer. How to read PnL?
"""

res = chat.invoke(query)
print(res['answer'], res['docs'])

I don't know. [Document(id='46fda176-7371-4384-acda-9b85b009634c', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='How to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw Hill Professional: Finance Made Easy Series\nTata McGraw Hill Professional: Finance Made Easy Series\nFinancial success is the raison d’être of any business, and fi nancial health\nof any organization is refl ected in its fi nancial statements. But, it has been\nobserved that managerial professionals often have little understanding of\nfi nance and little time to read treatises on it. Further, fi nancial statements\nare regarded as too complex to understand and left to be ‘deciphered’ by\nfi nance experts. Hence, cultivating a culture of awareness and transparency\nof fi nance is a prime imperative.\nFinance Made Easy Series has been designed to impart management\nexecutives with adequate knowledge to understand and appreciate fi nancial\nstatements and their implications for the fi scal solvency of the

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template(
    """
    Provide a better search query for web search engine to answer the question below,
    end the queries with "**". 
    Question: {question}
    Answer:
    """
)

def parse_rewrite_output(message):
    return message.split('**')[0]

rewrite = rewrite_prompt | llm | parse_rewrite_output

# rewrite.invoke(query)

@chain
def chat_rrr(query):
    rrr_query = rewrite.invoke(query)
    return chat.invoke(rrr_query)

res = chat_rrr.invoke(query)
print(res['answer'], res['docs'])



Here are key points about understanding a Profit and Loss (P&L) statement for beginners, based on the provided context:

*   **Purpose**: A P&L statement, also known as an Income Statement, summarizes a company's revenues and expenses over a specific period to determine its profit or loss.

*   **Importance**: It reflects the financial health of a company and helps stakeholders understand the profitability, compare performance over time, and predict future profits.

*   **Basic Calculation**: The basic formula is Revenues - Expenses = Profit (if revenues exceed expenses) or Loss (if expenses exceed revenues).

*   **Key Components**:
    *   **Revenue**: The income generated from sales of goods or services.
    *   **Cost of Goods Sold**: Direct costs associated with producing goods or delivering services.
    *   **Gross Profit**: Revenue minus the cost of goods sold.
    *   **Operating Expenses**: Expenses incurred during normal business operations (e.g., personnel, depreciation).
 

In [51]:
@chain
def chat_rrr(query):
    new_query = rewrite.invoke(query)
    
    docs = retriever.get_relevant_documents(new_query)

    formatted_docs = prompt_template.invoke({"question": new_query, "context": docs})

    return llm.invoke(formatted_docs)

print(query)    
print(chat_rrr.invoke(query))



Today I woke up and realized that my laptop battery was low.then I found a charger in the kitchen but it wasn't working properly.
So I decided to ask for help from my friend who is an engineer. How to read PnL?

I don't know


#### Multi-Query Retrieval to capture full scope of information required to answer the query.

In [29]:
from langchain.prompts import ChatPromptTemplate

perspectives_prompt = ChatPromptTemplate.from_template("""
                    You are an AI language model assistant. Your task is to generate five different version of the given question,
                    to retrieve relevant document from a vector database.By generating multiple perspectives of the user question,
                    your goal is to help the user overcome of the limitations of the distance-based similarity search in vector databases.
                    Provide these alternative questions separated by newlines DO NOT INCLUDE ANY OTHER TEXT just 5 similar queries only separated by \n. Original question: {question}
                   """)

def parse_queries_output(message):
    return message.split("\n")

query_gen = perspectives_prompt | llm | parse_queries_output


In [30]:
def get_unique_union(documents_list):
    # flatten the list and dedupe them
    deduped_docs = {
        doc.page_content: doc for sublist in documents_list for doc in sublist
    }

    return list(deduped_docs.values())

retrieval_chain = query_gen | retriever.batch | get_unique_union


In [31]:
## logger chain 

@chain 
def retrieval_chain_with_logger(question):
    raw_queries = query_gen.invoke(question)
    print(f"Generated queries: {raw_queries}")
    docs = retriever.batch(raw_queries)
    print(f"Retrieved documents: {docs}")

    summarized_docs = get_unique_union(docs)
    print(f"Unique documents: {summarized_docs}")

    return summarized_docs

In [32]:
prompt = ChatPromptTemplate.from_template("""
Answer the following question based on provided context:
{context}
Question: {question}
""")

@chain
def multi_query_retrieval(question):
    # docs = retrieval_chain.invoke(question)
    docs = retrieval_chain_with_logger.invoke(question)

    formatted = prompt.invoke({"context":docs,"question":question})

    answer = llm.invoke(formatted)

    return {"answer":answer, "source_documents":docs}



complete_answer = multi_query_retrieval.invoke("How to read the Profit and loss statement?")

print(complete_answer["answer"])
print(complete_answer["source_documents"])

Generated queries: ['What are the key components of a profit and loss statement and how do I interpret them?', "How can I analyze a company's financial performance using its income statement?", 'What are the common line items in a P&L statement and what do they tell me about a business?', 'Explain the process of understanding and interpreting a profit and loss statement.', 'What are some important ratios and metrics derived from the P&L statement and how are they calculated?']
Retrieved documents: [[Document(id='d936cd35-7dc8-4fc5-a811-1f67742a3f21', metadata={'source': 'data/how_to_read_pnl.txt'}, page_content='50 How to Read a Profi t and Loss Statement\nform part of the balance sheet). In Box 5.1, we uncover the essence of the\nrequirement of the income statement again.\nBox 5.1 Need for An Income Statement\nNormally, most of the events and transactions aff ect the assets\nand/or liabilities and hence aff ect the balance sheet. A change\nin the owner(s)’ equity can either be brought

<b> Small Exercise/Experiment </b>
<li> function to add document in db </li>
<li> query experiment </li>
<li> markdown into query </li>

In [ ]:
from langchain.indexes import SQLRecordManager, index
from langchain.docstore.document import Document
from langchain.vectorstores.pgvector import PGVector
from langchain.embeddings.base import Embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List
from langchain_community.document_loaders import PyMuPDFLoader

class RAGIndexer:
    def __init__(
        self,
        connection_string: str,
        embedding_model: Embeddings,
        user_id: str,
        mode: str = "private",
        base_collection: str = "rag_docs",
        use_jsonb: bool = True,
        source_key: str = "source"
    ):
        assert mode in ["private", "public"], "mode must be 'private' or 'public'"
        
        self.user_id = user_id
        self.mode = mode
        self.connection_string = connection_string
        self.embedding_model = embedding_model
        self.use_jsonb = use_jsonb
        self.source_key = source_key
        self.loader = None

        # Namespacing
        self.collection_name = f"{base_collection}_{user_id}" if mode == "private" else f"{base_collection}_public"
        self.namespace = f"{base_collection}_{user_id}" if mode == "private" else f"{base_collection}_public"

        # Record manager
        # self.record_manager = SQLRecordManager(namespace=self.namespace, db_url=connection_string)
        # self.record_manager.create_schema()

        # Vector store
        self.vector_store = PGVector(
            collection_name=self.collection_name,
            embedding=self.embedding_model,
            connection=connection_string,
            use_jsonb=use_jsonb
        )

        self._doc_filter = lambda x: bool(x.page_content and x.page_content.strip())

    def _add_metadata(self, doc: Document, visibility: str = "private") -> Document:
        metadata = doc.metadata or {}
        metadata["owner"] = self.user_id
        metadata["visibility"] = visibility
        doc.metadata = metadata
        return doc

    def _chunking(self, docs: List[Document]) -> List[Document]:
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        filtered_docs = [doc for doc in docs if self._doc_filter(doc)]
        return splitter.split_documents(filtered_docs)
    
    def load_pdf(self,path):
        if os.path.exists(path):
            print(f"Loading PDF from {path}")
            self.loader = PyMuPDFLoader(path)
            self.documents = self.loader.load()
        else:
            print("File not found at path", path)
            raise FileNotFoundError(f"File not found at {path}")
        
    def add_documents(self):
        if not self.documents:
            print("Documents are not loaded")
        else:
            print
            print("Documents are added successfully")



    # def add_documents(
    #     self,
    #     docs: List[Document],
    #     visibility: str = "private",
    #     source_id_key: str = "source"
    # ):
    #     enriched_docs = [self._add_metadata(doc, visibility) for doc in docs]
    #     chunked_docs = self._chunking(enriched_docs)

    #     return index(
    #         docs=chunked_docs,
    #         record_manager=self.record_manager,
    #         vector_store=self.vector_store,
    #         cleanup="incremental",
    #         source_id_key=source_id_key
    #     )

    def delete_documents_by_source_ids(self, source_ids: List[str]):
        for source_id in source_ids:
            self.record_manager.delete_keys([source_id])
            self.vector_store.delete(source_id)

    def search(
        self,
        query: str,
        k: int = 5,
        allow_public: bool = True
    ) -> List[Document]:
        results = self.vector_store.similarity_search(query, k=k)

        if allow_public and self.mode != "public":
            public_vs = PGVector.from_existing_index(
                collection_name="rag_docs_public",
                embedding=self.embedding_model,
                connection=self.connection_string,
                use_jsonb=self.use_jsonb
            )
            results.extend(public_vs.similarity_search(query, k=k))

        return results

    def get_vector_store(self):
        return self.vector_store

    def get_record_manager(self):
        return self.record_manager


In [50]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.docstore.document import Document

embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GoogleGeminiKey)

indexer = RAGIndexer(
    connection_string=connection,
    embedding_model=embedding_model,
    user_id="diljit_647",
    mode="private"
)

docs = [
    Document(page_content="This is a private finance report.", metadata={"source": "finance.pdf"}),
    Document(page_content="Public climate change article.", metadata={"source": "climate.txt"})
]

# Add one private and one public doc
indexer.add_documents([docs[0]], visibility="private")
indexer.add_documents([docs[1]], visibility="public")

# Search
results = indexer.search("climate change", k=3, allow_public=True)

for doc in results:
    print(f"{doc.page_content} — Metadata: {doc.metadata}")


f:\miniconda\envs\vlang\Lib\site-packages\langchain_community\vectorstores\pgvector.py:1092: LangChainPendingDeprecationWarning: Please use JSONB instead of JSON for metadata. This change will allow for more efficient querying that involves filtering based on metadata. Please note that filtering operators have been changed when using JSONB metadata to be prefixed with a $ sign to avoid name collisions with columns. If you're using an existing database, you will need to create a db migration for your metadata column to be JSONB and update your queries to use the new operators. 
  store = cls(


[Document(metadata={'source': 'finance.pdf', 'owner': 'diljit_647', 'visibility': 'private'}, page_content='This is a private finance report.')]


ProgrammingError: (psycopg.errors.UndefinedColumn) column "custom_id" of relation "langchain_pg_embedding" does not exist
LINE 1: ...g (collection_id, embedding, document, cmetadata, custom_id,...
                                                             ^
[SQL: INSERT INTO langchain_pg_embedding (collection_id, embedding, document, cmetadata, custom_id, uuid) VALUES (%(collection_id)s::UUID, %(embedding)s, %(document)s::VARCHAR, %(cmetadata)s::JSON, %(custom_id)s::VARCHAR, %(uuid)s::UUID)]
[parameters: [{'collection_id': UUID('13943524-da7e-4a7a-8b88-2114e303d7e1'), 'embedding': '[0.03640071302652359,-0.06329690665006638,-0.03448600322008133,-0.009886731393635273,0.10148289799690247,0.008421013131737709,0.03874686732888222,0.0 ... (15899 characters truncated) ... 026335477828979492,-5.8144520153291523e-05,0.016901932656764984,0.045932404696941376,0.06183097884058952,0.0012061718152835965,-0.011265788227319717]', 'document': 'How to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw H

In [52]:
indexer.add_documents(
    docs=[
        Document(page_content="Here's a private contract.", metadata={"source": "contract.pdf"}),
        Document(page_content="Public news article on tech.", metadata={"source": "news.txt"})
    ],
    visibility="private"
)

results = indexer.search("contract law")

[Document(metadata={'source': 'contract.pdf', 'owner': 'diljit_647', 'visibility': 'private'}, page_content="Here's a private contract."), Document(metadata={'source': 'news.txt', 'owner': 'diljit_647', 'visibility': 'private'}, page_content='Public news article on tech.')]


ProgrammingError: (psycopg.errors.UndefinedColumn) column "custom_id" of relation "langchain_pg_embedding" does not exist
LINE 1: ...g (collection_id, embedding, document, cmetadata, custom_id,...
                                                             ^
[SQL: INSERT INTO langchain_pg_embedding (collection_id, embedding, document, cmetadata, custom_id, uuid) VALUES (%(collection_id)s::UUID, %(embedding)s, %(document)s::VARCHAR, %(cmetadata)s::JSON, %(custom_id)s::VARCHAR, %(uuid)s::UUID)]
[parameters: [{'collection_id': UUID('13943524-da7e-4a7a-8b88-2114e303d7e1'), 'embedding': '[0.03640071302652359,-0.06329690665006638,-0.03448600322008133,-0.009886731393635273,0.10148289799690247,0.008421013131737709,0.03874686732888222,0.0 ... (15899 characters truncated) ... 026335477828979492,-5.8144520153291523e-05,0.016901932656764984,0.045932404696941376,0.06183097884058952,0.0012061718152835965,-0.011265788227319717]', 'document': 'How to Read\nA\nPROFIT AND LOSS\nSTATEMENT\nTata McGraw H